# JSON 輸出與 Structured Outputs

## 模組脈絡：從「看起來像 JSON」收斂到「保證符合 schema」

本筆記是 **03-結構化輸出** 的起點。2026 新專案應以 **Responses API** 為主：

- 普通 prompt 只能要求模型「盡量輸出 JSON」。
- JSON mode 只保證輸出是可解析 JSON，不保證欄位或型別。
- **Structured Outputs** 透過 JSON Schema / Pydantic 保證 schema adherence，是需要型別化資料時的首選。

官方遷移重點：Responses API 的結構化輸出使用 `text.format`；Python SDK 也提供 `client.responses.parse(..., text_format=Model)`。舊 Chat Completions 參數只應出現在遷移對照，不應作為新專案主線。

## 0. 環境設定

In [ ]:
from dotenv import load_dotenv
import os
import json

from openai import OpenAI
from pydantic import BaseModel

load_dotenv()
client = OpenAI()  # 讀取 OPENAI_API_KEY
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")

## 1. 普通文字生成：要求 JSON，但不保證格式

只在 prompt 寫「請用 JSON」仍可能出現 Markdown code fence、額外說明、缺欄位或型別錯誤。這是 prompt-level 約束，不是 API-level 約束。

In [ ]:
response = client.responses.create(
    model=OPENAI_MODEL,
    input=[{"role": "user", "content": "請隨機產生三個台灣 user 資料，請用 JSON 格式回傳。"}],
    temperature=0.2,
    max_output_tokens=500,
)
print(response.output_text)

## 2. JSON mode：保證可解析 JSON，但不保證 schema

JSON mode 適合快速取得合法 JSON。它不會保證欄位完整、型別正確或 enum 合法；需要這些保證時，應改用 Structured Outputs。

In [ ]:
response = client.responses.create(
    model=OPENAI_MODEL,
    input=[
        {"role": "system", "content": "You are a helpful assistant designed to output JSON."},
        {"role": "user", "content": "請產生三個台灣 user 資料，格式為 {users: [...]}."},
    ],
    text={"format": {"type": "json_object"}},
    temperature=0.2,
    max_output_tokens=500,
)

raw_json = response.output_text
print(raw_json)
print(json.loads(raw_json))

## 3. Structured Outputs：用 Pydantic 取得型別化結果

`client.responses.parse(..., text_format=Model)` 會讓 SDK 把 Pydantic model 轉成 schema，並將結果解析到 `response.output_parsed`。這是教學與實務都最清楚的 2026 寫法。

In [ ]:
class User(BaseModel):
    name: str
    age: int
    bio: str
    avatar_url: str
    is_subscriber: bool


class Users(BaseModel):
    users: list[User]


response = client.responses.parse(
    model=OPENAI_MODEL,
    input=[
        {"role": "system", "content": "Generate realistic structured data for a Taiwan product demo."},
        {"role": "user", "content": "隨機產生三個台灣使用者資料。"},
    ],
    text_format=Users,
)

users = response.output_parsed
print(users)
print(users.users[0].name, type(users.users[0].age))

## 4. JSON Schema 版：跨語言或不想依賴 Pydantic 時使用

如果團隊不是 Python，或 schema 來自 API contract，可以直接傳 JSON Schema。注意 `strict=True` 只支援 JSON Schema 的子集合；schema 應保持明確、簡潔。

In [ ]:
user_schema = {
    "type": "object",
    "additionalProperties": False,
    "properties": {
        "users": {
            "type": "array",
            "items": {
                "type": "object",
                "additionalProperties": False,
                "properties": {
                    "name": {"type": "string"},
                    "age": {"type": "integer"},
                    "is_subscriber": {"type": "boolean"},
                },
                "required": ["name", "age", "is_subscriber"],
            },
        }
    },
    "required": ["users"],
}

response = client.responses.create(
    model=OPENAI_MODEL,
    input="隨機產生三個台灣使用者資料。",
    text={
        "format": {
            "type": "json_schema",
            "name": "users_payload",
            "strict": True,
            "schema": user_schema,
        }
    },
    temperature=0.2,
    max_output_tokens=500,
)

print(response.output_text)
print(json.loads(response.output_text))

## JSON mode vs Function Calling vs Structured Outputs 怎麼選

| 方法 | schema 強制力 | 適用 | 2026 Responses 寫法 |
|------|--------------|------|---------------------|
| Prompt-only JSON | 無 | demo、草稿、人工閱讀 | `client.responses.create(..., input="請用 JSON")` |
| JSON mode | 弱，只保證合法 JSON | 快速串接、結構很簡單 | `text={"format": {"type": "json_object"}}` |
| **Structured Outputs** | **強，保證符合 schema** | 型別化資料、可驗證輸出、UI/API contract | `client.responses.parse(..., text_format=Model)` 或 `text.format=json_schema` |
| Function/Tool calling | 工具參數強約束 | 要呼叫外部工具或系統動作 | `tools=[...]`，見後續 function calling 章節 |

原則：如果資料會進資料庫、API、UI 或評估流程，直接用 Structured Outputs，不要只靠 JSON mode。

---

## 本章小結

1. Prompt-only JSON 只是文字約束，容易混入 Markdown 或漏欄位。
2. JSON mode 只保證**可解析 JSON**，不保證 schema。
3. Structured Outputs 是 2026 首選：Responses API 使用 `text.format`；Python SDK 可用 `client.responses.parse(..., text_format=Model)`。
4. 工具呼叫與結構化輸出不是同一件事：要回傳資料用 Structured Outputs，要執行外部動作用 tools/function calling。
5. 模型由 `OPENAI_MODEL` 控制，避免教材綁死單一模型。